# 🌲 Step 4 — ML Species Classification
**Forest Carbon Continuity Pipeline · Sentinel-1/2 Hybrid for EU LULUCF Compliance**

## What this step does
Builds a synthetic 6-band feature stack (radar + optical + texture), trains a **Random Forest**
to classify three species — Sitka Spruce, Lodgepole Pine, Native Mixed — and reports accuracy,
per-class precision/recall, a confusion matrix, feature importances, and the full predicted
species map.

## The Science
Species matters because each species carries a **different biomass factor** (Step 5). The
production architecture in `src/ml_models.py` is a **U-Net with Dice+BCE loss** (spatial
segmentation), exactly like DIGIGAS keeps GNN/LSTM as the follow-on — the day-one prototype here
uses a Random Forest on per-pixel features, which trains in seconds on Colab CPU and needs no GPU.
The 6 features mirror what Steps 2-3 produce in production: VH dB, VH/VV ratio, NDVI, Red, NIR,
and a texture proxy. Swap in real features later **without changing the modeling logic**.

In [ ]:
# ============================================================
# Environment detection & package setup (runs first, auto-detects
# Google Colab vs local Jupyter - nothing needs manual pip install)
# ============================================================
import sys, subprocess

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    colab_files = None

_ensure("numpy"); _ensure("pandas"); _ensure("matplotlib")
_ensure("rasterio"); _ensure("geopandas"); _ensure("scikit-learn")
_ensure("openpyxl"); _ensure("ipywidgets")

def outpath(name):
    """Returns a working save path for the current environment."""
    return f"/content/{name}" if IN_COLAB else name

print(f"Environment detected: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")

# ============================================================
# Unified cross-environment helpers:
# download buttons, regenerate buttons, upload widgets
# ============================================================
import zipfile
import ipywidgets as widgets
from IPython.display import display, FileLink, clear_output

_generated_files = {}  # friendly label -> filepath, used by the final "Download All" bundle

def register_file(label, path):
    if label:
        _generated_files[label] = path

def make_download_button(path, label=None, register_as=None):
    """Cross-environment download button. Colab: real browser download.
    Local Jupyter: clickable FileLink. Reads the file fresh at click time."""
    label = label or f"Download {path}"
    if register_as:
        register_file(register_as, path)
    btn = widgets.Button(description=label, button_style="info", icon="download")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            if IN_COLAB:
                colab_files.download(path)
            else:
                display(FileLink(path))
                print(f"Click the link above to download '{path}'.")

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_regenerate_button(rerun_fn, label="🔄 Regenerate & Re-run"):
    """Button that calls rerun_fn() (fresh random seed, full re-run of this
    section) and shows the refreshed output inline below the button."""
    btn = widgets.Button(description=label, button_style="warning", icon="refresh")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            rerun_fn()

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_upload_widget(on_loaded, accept=".csv,.xlsx", label="Upload replacement file"):
    """Colab: native blocking file picker via google.colab.files.upload().
    Local Jupyter: ipywidgets.FileUpload + Load button (non-blocking)."""
    out = widgets.Output()

    if IN_COLAB:
        btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                print("Choose a file in the dialog, or Cancel to keep current data.")
                uploaded = colab_files.upload()
                if uploaded:
                    fname = next(iter(uploaded))
                    on_loaded(fname, uploaded[fname])
                else:
                    print("No file selected - keeping current data.")

        btn.on_click(on_click)
        display(widgets.VBox([btn, out]))
    else:
        uploader = widgets.FileUpload(accept=accept, multiple=False, description="Choose file")
        load_btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                val = uploader.value
                if not val:
                    print("No file selected yet - choose a file above, then click the button.")
                    return
                item = list(val.values())[0] if isinstance(val, dict) else val[0]
                if isinstance(item, dict) and "metadata" in item:
                    fname = item["metadata"]["name"]     # ipywidgets 7.x shape
                else:
                    fname = item["name"]                  # ipywidgets 8.x shape
                on_loaded(fname, bytes(item["content"]))

        load_btn.on_click(on_click)
        display(widgets.VBox([uploader, load_btn, out]))

def make_download_all_button(label="⬇️ Download ALL outputs (.zip)"):
    """Zips every output registered so far and offers it as one download."""
    zip_path = outpath("outputs_bundle.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for _label, _path in _generated_files.items():
            try:
                zf.write(_path)
            except FileNotFoundError:
                pass
    make_download_button(zip_path, label=label)

print("Helpers ready: make_download_button / make_regenerate_button / make_upload_widget")

In [ ]:
# ============================================================
# Standalone synthetic feature stack + species labels (Day-1 ready)
# ============================================================
import numpy as np
import rasterio
from rasterio.transform import from_bounds

H, W = 120, 120
BOUNDS = (-6.4, 52.95, -6.1, 53.15)
TRANSFORM = from_bounds(*BOUNDS, W, H)
FEATURE_NAMES = ["VH_dB", "VH_VV_ratio", "NDVI", "Red", "NIR", "Texture"]
SPECIES_NAMES = {1: "Sitka Spruce", 2: "Lodgepole Pine", 3: "Native Mixed"}

def generate_feature_stack(seed=11):
    """6-band synthetic feature stack with three smooth species zones."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:H, 0:W]

    def blob(cx, cy, sx, sy):
        return np.exp(-(((xx - cx) / sx) ** 2 + ((yy - cy) / sy) ** 2))

    b_spruce = blob(35, 40, 22, 18)
    b_pine = blob(85, 85, 26, 22)
    b_mixed = blob(80, 25, 20, 20)
    labels = (np.argmax(np.stack([b_spruce, b_pine, b_mixed]), axis=0) + 1).astype("uint8")

    features = np.stack([
        -13 + 3.5 * b_spruce + 1.2 * b_mixed + rng.normal(0, 0.8, (H, W)),        # VH dB (spruce densest)
        0.20 + 0.14 * b_pine + rng.normal(0, 0.015, (H, W)),                      # VH/VV ratio (pine signature)
        0.75 + 0.12 * b_spruce - 0.06 * b_pine - 0.15 * b_mixed
        + rng.normal(0, 0.04, (H, W)),                                            # NDVI (mixed lower)
        0.20 + 0.06 * b_mixed + rng.normal(0, 0.03, (H, W)),                      # Red (mixed higher)
        0.60 + 0.10 * b_spruce - 0.05 * b_mixed + rng.normal(0, 0.06, (H, W)),    # NIR
        0.30 + 0.30 * b_mixed + rng.normal(0, 0.04, (H, W)),                      # Texture (mixed = heterogeneous)
    ]).astype("float32")
    return features, labels

features, labels = generate_feature_stack()
print(f"Feature stack: {features.shape[0]} bands x {H}x{W} | species zones: 1/2/3")

In [ ]:
# ============================================================
# Train + evaluate the Random Forest species classifier
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def run_classification(features, labels, tag="synthetic"):
    X = features.reshape(features.shape[0], -1).T          # (pixels, bands)
    y = labels.ravel()
    idx = np.arange(len(y))
    train_idx, test_idx = train_test_split(idx, test_size=0.25,
                                           random_state=42, stratify=y)

    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X[train_idx], y[train_idx])
    y_pred = model.predict(X[test_idx])

    acc = accuracy_score(y[test_idx], y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y[test_idx], y_pred, labels=[1, 2, 3], zero_division=0)
    print(f"Overall accuracy: {acc*100:.1f}%")
    metrics = pd.DataFrame({
        "Species": [SPECIES_NAMES[i] for i in [1, 2, 3]],
        "Precision": prec.round(3), "Recall": rec.round(3), "F1": f1.round(3),
    })
    print(metrics.to_string(index=False))

    pred_map = model.predict(X).reshape(H, W).astype("uint8")
    cm = confusion_matrix(y[test_idx], y_pred, labels=[1, 2, 3])

    fig, axes = plt.subplots(2, 2, figsize=(11.5, 9))
    axes[0, 0].imshow(labels, cmap="terrain", vmin=1, vmax=3)
    axes[0, 0].set_title("True species zones (synthetic ground truth)")
    axes[0, 1].imshow(pred_map, cmap="terrain", vmin=1, vmax=3)
    axes[0, 1].set_title("Random Forest prediction (all pixels)")
    im = axes[1, 0].imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        axes[1, 0].text(j, i, str(v), ha="center", va="center")
    names = [f"{k}. {SPECIES_NAMES[k]}" for k in [1, 2, 3]]
    axes[1, 0].set_xticks([0, 1, 2]); axes[1, 0].set_xticklabels(names, rotation=20, fontsize=7)
    axes[1, 0].set_yticks([0, 1, 2]); axes[1, 0].set_yticklabels(names, fontsize=7)
    axes[1, 0].set_title("Confusion matrix (test set)")
    importances = pd.Series(model.feature_importances_, index=FEATURE_NAMES).sort_values()
    axes[1, 1].barh(importances.index, importances.values, color="#2e7d32")
    axes[1, 1].set_title("Feature importance")
    plt.tight_layout()

    chart_path = outpath("step4_classification_chart.png")
    fig.savefig(chart_path, dpi=120, bbox_inches="tight")
    plt.show()

    map_path = outpath("step4_species_map.tif")
    with rasterio.open(map_path, "w", driver="GTiff", height=H, width=W, count=1,
                       dtype="uint8", crs="EPSG:4326", transform=TRANSFORM) as dst:
        dst.write(pred_map, 1)

    metrics_path = outpath("step4_classification_metrics.xlsx")
    with pd.ExcelWriter(metrics_path, engine="openpyxl") as writer:
        metrics.to_excel(writer, sheet_name="Per_Species_Metrics", index=False)
        pd.DataFrame({"Metric": ["overall_accuracy"],
                      "Value": [round(acc, 4)]}).to_excel(
            writer, sheet_name="Summary", index=False)
        importances.rename("importance").to_excel(writer, sheet_name="Feature_Importance")

    make_download_button(chart_path, label="⬇️ Download classification chart (.png)",
                         register_as=f"Step 4 chart ({tag})")
    make_download_button(map_path, label="⬇️ Download species map (.tif)",
                         register_as=f"Step 4 species map ({tag})")
    make_download_button(metrics_path, label="⬇️ Download metrics (.xlsx)",
                         register_as=f"Step 4 metrics ({tag})")
    return model, pred_map

model, pred_map = run_classification(features, labels)

**Interpretation:** the prediction panel (top-right) reproduces the true zones (top-left) with
fuzzy edges where species signatures overlap — exactly where a real classifier struggles too. The
confusion matrix shows **which species get swapped** (off-diagonal cells), and the feature-importance
panel tells you *why*: in this synthetic world VH dB and NDVI carry the species signal. In
production the same table is your first diagnostic: if a real-world run shows species collapsing
into one class, the feature engineering — not the model — is usually the culprit.

In [ ]:
# ============================================================
# Optional: upload your own 6-band feature stack (.tif, band order:
# VH_dB, VH_VV_ratio, NDVI, Red, NIR, Texture) - labels stay synthetic
# ============================================================
def handle_stack_upload(fname, content_bytes):
    tmp = outpath("uploaded_feature_stack.tif")
    open(tmp, "wb").write(content_bytes)
    try:
        with rasterio.open(tmp) as src:
            if src.count != len(FEATURE_NAMES):
                print(f"❌ Upload rejected - '{fname}' has {src.count} band(s); need {len(FEATURE_NAMES)}.")
                return
            u_feats = src.read().astype("float32")
    except Exception as e:
        print(f"❌ Could not read '{fname}': {e}")
        return
    print(f"✅ SUCCESS: loaded {u_feats.shape[0]}-band stack from '{fname}'. Re-training...")
    run_classification(u_feats, labels, tag=f"uploaded:{fname}")

make_upload_widget(handle_stack_upload, accept=".tif,.tiff", label="Upload 6-band stack (.tif)")
print("Skip to continue with the synthetic stack.")

# ============================================================
# Regenerate control
# ============================================================
def rerun_step4(seed=None):
    """Fresh synthetic stack with a new seed + full retrain and re-plot."""
    global features, labels, pred_map
    seed = seed or np.random.randint(0, 10_000)
    features, labels = generate_feature_stack(seed=seed)
    print(f"[seed={seed}] regenerated feature stack")
    model, pred_map = run_classification(features, labels, tag=f"seed-{seed}")

make_regenerate_button(rerun_step4, label="🔄 Regenerate stack & re-train")
make_download_all_button()